# Entrenamiento YOLOv8n — INDIGO Crack Detection (Google Colab)

Notebook reproducible para entrenar el **baseline** de detección con Ultralytics YOLOv8n
sobre el dataset INDIGO ya convertido a formato YOLO.

**Requisitos previos**

- Subir `data/processed/indigo_yolo/` como ZIP a Google Drive:
  `/content/drive/MyDrive/egg-detection/datasets/indigo_yolo.zip`
- Runtime con **GPU** (T4 o superior recomendado).

**Estrategia de I/O**

- **Dataset:** ZIP persistente en Drive → extracción temporal a `/content/indigo_yolo/`
  (lectura rápida durante entrenamiento).
- **Resultados:** checkpoints y métricas en Drive
  (`/content/drive/MyDrive/egg-detection/runs/yolov8n_baseline/`).

**Reglas de evaluación**

- `val` se usa durante el entrenamiento y para métricas de validación.
- `test` se evalúa **una sola vez** al final; no usar para ajustar hiperparámetros.

Documentación: [`docs/16-entrenamiento-yolov8n.md`](../docs/16-entrenamiento-yolov8n.md)

## 1. Verificar entorno

Comprobar Python, GPU disponible y memoria.

In [2]:
import platform
import subprocess
import sys

print("Python:", sys.version)
print("Plataforma:", platform.platform())

try:
    import torch

    cuda_ok = torch.cuda.is_available()
    print("PyTorch:", torch.__version__)
    print("CUDA disponible:", cuda_ok)
    if cuda_ok:
        print("GPU:", torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print(f"Memoria GPU total: {props.total_memory / (1024**3):.2f} GiB")
    else:
        print("\n⚠️  ADVERTENCIA: No hay GPU disponible.")
        print("   Activa Runtime → Change runtime type → GPU antes de entrenar.")
        print("   El entrenamiento en CPU será extremadamente lento.")
except ImportError:
    print("PyTorch aún no instalado (se instalará con Ultralytics).")

print("\n--- nvidia-smi ---")
try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi no disponible (¿runtime sin GPU?)")

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Plataforma: Windows-10-10.0.26200-SP0
PyTorch: 2.14.0+cpu
CUDA disponible: False

⚠️  ADVERTENCIA: No hay GPU disponible.
   Activa Runtime → Change runtime type → GPU antes de entrenar.
   El entrenamiento en CPU será extremadamente lento.

--- nvidia-smi ---


## 2. Instalar dependencias

Solo Ultralytics y sus dependencias transitivas.

In [3]:
!pip install -q ultralytics

import ultralytics

print("Ultralytics:", ultralytics.__version__)


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Migue\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics: 8.4.157


## 3. Montar Google Drive

El dataset vive como ZIP en Drive. El entrenamiento leerá desde `/content/` (no desde Drive).
Ajusta `DATASET_ZIP` si subiste el archivo a otra ruta.

In [4]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# --- Rutas configurables ---
DATASET_ZIP = Path("/content/drive/MyDrive/egg-detection/datasets/indigo_yolo.zip")
DATASET_ROOT = Path("/content/indigo_yolo")  # extracción local (I/O rápido)
RUNS_ROOT = Path("/content/drive/MyDrive/egg-detection/runs")
RUN_NAME = "yolov8n_baseline"

print("DATASET_ZIP:", DATASET_ZIP)
print("  existe:", DATASET_ZIP.exists())
print("DATASET_ROOT (local):", DATASET_ROOT)
print("RUNS_ROOT (Drive):", RUNS_ROOT)

ModuleNotFoundError: No module named 'google.colab'

## 4. Extraer dataset a `/content` y verificar

1. Comprobar que el ZIP existe y hay espacio libre en `/content`.
2. Extraer a `/content/indigo_yolo/` (omitir si ya está válido).
3. Verificar conteos: **629 train**, **111 val**, **100 test**.
4. Generar `data_colab.yaml` con rutas de Colab.

In [ ]:
import shutil
import zipfile

import yaml

EXPECTED = {"train": 629, "val": 111, "test": 100}
REQUIRED_DIRS = [
    "images/train",
    "images/val",
    "images/test",
    "labels/train",
    "labels/val",
    "labels/test",
]


def contar_dataset(root: Path) -> dict[str, dict[str, int]]:
    counts = {}
    for split in EXPECTED:
        n_img = len(list((root / "images" / split).glob("*.jpg")))
        n_lbl = len(list((root / "labels" / split).glob("*.txt")))
        counts[split] = {"images": n_img, "labels": n_lbl}
    return counts


def validar_conteos(counts: dict[str, dict[str, int]]) -> list[str]:
    errors = []
    for split, exp in EXPECTED.items():
        if counts[split]["images"] != exp:
            errors.append(f"{split}: imágenes={counts[split]['images']} (esperado {exp})")
        if counts[split]["labels"] != exp:
            errors.append(f"{split}: labels={counts[split]['labels']} (esperado {exp})")
    return errors


def dataset_ya_valido(root: Path) -> bool:
    if not root.exists():
        return False
    missing = [d for d in REQUIRED_DIRS if not (root / d).exists()]
    if missing:
        return False
    return len(validar_conteos(contar_dataset(root))) == 0


def encontrar_raiz_yolo(base: Path) -> Path:
    if (base / "images" / "train").is_dir():
        return base
    candidato = base / "indigo_yolo"
    if (candidato / "images" / "train").is_dir():
        return candidato
    for child in base.iterdir():
        if child.is_dir() and (child / "images" / "train").is_dir():
            return child
    raise FileNotFoundError(
        "No se encontró estructura YOLO (images/train, labels/train, ...) tras extraer el ZIP."
    )


def extraer_desde_zip(zip_path: Path, destino: Path) -> None:
    tmp = Path("/content/_indigo_extract_tmp")
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True)

    print("Extrayendo ZIP →", tmp)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(tmp)

    raiz = encontrar_raiz_yolo(tmp)
    if destino.exists():
        shutil.rmtree(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(raiz), str(destino))
    shutil.rmtree(tmp, ignore_errors=True)
    print("Dataset listo en:", destino)


if dataset_ya_valido(DATASET_ROOT):
    print("✅ Dataset ya extraído y válido en /content — omitiendo extracción.")
else:
    if not DATASET_ZIP.exists():
        raise FileNotFoundError(
            f"No se encontró el ZIP del dataset: {DATASET_ZIP}\n"
            "Sube indigo_yolo.zip a Google Drive en esa ruta."
        )

    zip_bytes = DATASET_ZIP.stat().st_size
    _, used, free = shutil.disk_usage("/content")
    # Margen: ZIP + ~2.5 GiB descomprimido (dataset ≈ 2.24 GiB)
    espacio_requerido = max(int(zip_bytes * 1.2), int(2.6 * 1024**3))

    print(f"Tamaño ZIP: {zip_bytes / (1024**3):.3f} GiB ({zip_bytes:,} bytes)")
    print(f"Espacio libre en /content: {free / (1024**3):.3f} GiB")
    print(f"Espacio mínimo requerido (estimado): {espacio_requerido / (1024**3):.3f} GiB")

    if free < espacio_requerido:
        raise RuntimeError(
            "Espacio insuficiente en /content para extraer el dataset. "
            "Libera espacio o reinicia el runtime de Colab."
        )

    extraer_desde_zip(DATASET_ZIP, DATASET_ROOT)

# --- Verificación post-extracción ---
missing = [d for d in REQUIRED_DIRS if not (DATASET_ROOT / d).exists()]
if missing:
    raise FileNotFoundError(
        "Faltan carpetas del dataset:\n  " + "\n  ".join(str(DATASET_ROOT / d) for d in missing)
    )

counts = contar_dataset(DATASET_ROOT)
print("\nConteos encontrados:")
for split, c in counts.items():
    print(f"  {split}: {c['images']} imgs, {c['labels']} labels (esperado {EXPECTED[split]})")

errors = validar_conteos(counts)
if errors:
    raise ValueError("Conteos del dataset no coinciden:\n" + "\n".join(errors))

print("\n✅ Dataset verificado correctamente.")

# data.yaml original puede tener rutas Windows — usar data_colab.yaml
DATA_YAML = DATASET_ROOT / "data_colab.yaml"
yaml_payload = {
    "path": "/content/indigo_yolo",
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 2,
    "names": {0: "egg", 1: "crack"},
}
DATA_YAML.write_text(yaml.dump(yaml_payload, default_flow_style=False), encoding="utf-8")
print("data_colab.yaml:", DATA_YAML)

orig_yaml = DATASET_ROOT / "data.yaml"
if orig_yaml.exists():
    print("(data.yaml original presente; se usa data_colab.yaml para entrenamiento)")
else:
    print("(data.yaml original no incluido en ZIP; data_colab.yaml es suficiente)")

## 5. Configuración reproducible

Semilla fija `42`. Los resultados exactos pueden variar ligeramente según GPU,
versión de CUDA y operaciones no deterministas en PyTorch.

In [ ]:
import os
import random

import numpy as np
import torch

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Determinismo razonable (puede reducir velocidad)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("SEED:", SEED)
print("Nota: métricas finales pueden variar levemente entre ejecuciones/hardware.")

## 6. Modelo baseline

Cargar pesos preentrenados **YOLOv8n** (nano). No usar YOLOv8s en este baseline.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
print("Modelo baseline:", model.model_name if hasattr(model, "model_name") else "yolov8n")

## 7. Entrenamiento

Hiperparámetros conservadores para un baseline inicial. Resultados en Drive.

In [ ]:
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError(
        "No hay GPU. Activa GPU en Colab antes de entrenar (Runtime → Change runtime type)."
    )

train_results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=-1,          # auto: Ultralytics elige batch seguro para la GPU
    patience=10,
    seed=SEED,
    device=0,
    workers=2,
    project=str(RUNS_ROOT),
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    verbose=True,
)

RUN_DIR = RUNS_ROOT / RUN_NAME
print("Directorio de resultados:", RUN_DIR)

## 8. Checkpoints

Verificar que `best.pt` y `last.pt` existen en Google Drive.

In [ ]:
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"

for path in (BEST_PT, LAST_PT):
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"✅ {path.name}: {path} ({size_mb:.1f} MiB)")
    else:
        raise FileNotFoundError(f"Checkpoint no encontrado: {path}")

# Recargar el mejor checkpoint para evaluación
model = YOLO(str(BEST_PT))

## 9. Métricas de validación

Evaluar sobre el split **val**. Métricas extraídas directamente de Ultralytics
(no inventadas).

In [ ]:
val_metrics = model.val(
    data=str(DATA_YAML),
    split="val",
    device=0,
    plots=True,
    save_json=True,
)


def extract_box_metrics(metrics):
    """Extrae precision, recall, mAP50 y mAP50-95 de resultados Ultralytics."""
    box = metrics.box
    return {
        "precision": float(box.mp),
        "recall": float(box.mr),
        "mAP50": float(box.map50),
        "mAP50_95": float(box.map),
    }


val_scores = extract_box_metrics(val_metrics)

print("=== Métricas VALIDATION (val) ===")
for k, v in val_scores.items():
    print(f"  {k}: {v:.4f}")

## 10. Test final (una sola vez)

⚠️ **Evaluación final reservada.** El split `test` no se usó para entrenar ni
para early stopping. Se evalúa **una única vez** aquí.

In [ ]:
test_metrics = model.val(
    data=str(DATA_YAML),
    split="test",
    device=0,
    plots=True,
    save_json=True,
)

test_scores = extract_box_metrics(test_metrics)

print("=== Métricas TEST FINAL (evaluación única) ===")
for k, v in test_scores.items():
    print(f"  {k}: {v:.4f}")

## 11. Matriz de confusión y curvas

Mostrar gráficas generadas por Ultralytics durante la validación.

In [ ]:
from IPython.display import Image as IPyImage, display

plot_candidates = [
    RUN_DIR / "confusion_matrix.png",
    RUN_DIR / "confusion_matrix_normalized.png",
    RUN_DIR / "PR_curve.png",
    RUN_DIR / "F1_curve.png",
    RUN_DIR / "P_curve.png",
    RUN_DIR / "R_curve.png",
]

# Ultralytics también puede guardar plots en subcarpetas de val
for extra in RUN_DIR.rglob("*.png"):
    name = extra.name.lower()
    if any(k in name for k in ("confusion", "pr_curve", "f1_curve", "p_curve", "r_curve")):
        if extra not in plot_candidates:
            plot_candidates.append(extra)

shown = set()
for plot_path in plot_candidates:
    if plot_path.exists() and plot_path not in shown:
        shown.add(plot_path)
        print(plot_path.name)
        display(IPyImage(filename=str(plot_path)))

## 12. Curvas de entrenamiento

`results.png` y métricas por época (losses, precision, recall, mAP).

In [ ]:
results_png = RUN_DIR / "results.png"
if results_png.exists():
    print("results.png")
    display(IPyImage(filename=str(results_png)))
else:
    print("results.png no encontrado en", RUN_DIR)

results_csv = RUN_DIR / "results.csv"
if results_csv.exists():
    import pandas as pd

    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    display(df.tail(10))

## 13. Predicciones visuales (test)

Inferencia con `best.pt` sobre imágenes aleatorias del test. No se modifican
las imágenes originales.

In [ ]:
import matplotlib.pyplot as plt

test_img_dir = DATASET_ROOT / "images" / "test"
test_images = sorted(test_img_dir.glob("*.jpg"))

rng = random.Random(SEED)
n_samples = min(6, len(test_images))
sample_paths = rng.sample(test_images, n_samples)

predictions = model.predict(
    source=[str(p) for p in sample_paths],
    conf=0.25,
    save=False,
    verbose=False,
)

for src_path, result in zip(sample_paths, predictions):
    annotated = result.plot()  # BGR numpy
    plt.figure(figsize=(8, 10))
    plt.imshow(annotated[:, :, ::-1])
    plt.title(src_path.name)
    plt.axis("off")
    plt.show()

    classes = [result.names[int(c)] for c in result.boxes.cls.tolist()] if result.boxes is not None else []
    print(f"  {src_path.name} → detectado: {classes if classes else '(sin detecciones)'}")

## 14. Exportar resumen de resultados

Guardar JSON y CSV en Google Drive con métricas reales de la ejecución.

In [ ]:
import csv
import json
from datetime import datetime, timezone

summary = {
    "modelo": "yolov8n",
    "fecha_utc": datetime.now(timezone.utc).isoformat(),
    "epochs": 50,
    "imgsz": 640,
    "batch": "auto (-1)",
    "patience": 10,
    "seed": SEED,
    "dataset_zip": str(DATASET_ZIP),
    "dataset_root": str(DATASET_ROOT),
    "data_yaml": str(DATA_YAML),
    "precision_val": val_scores["precision"],
    "recall_val": val_scores["recall"],
    "mAP50_val": val_scores["mAP50"],
    "mAP50_95_val": val_scores["mAP50_95"],
    "precision_test": test_scores["precision"],
    "recall_test": test_scores["recall"],
    "mAP50_test": test_scores["mAP50"],
    "mAP50_95_test": test_scores["mAP50_95"],
    "best_pt": str(BEST_PT),
    "last_pt": str(LAST_PT),
    "run_dir": str(RUN_DIR),
    "ultralytics_version": ultralytics.__version__,
}

summary_json = RUN_DIR / "training_summary.json"
summary_csv = RUN_DIR / "training_summary.csv"

summary_json.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

with summary_csv.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(summary.keys()))
    writer.writeheader()
    writer.writerow(summary)

print("Resumen guardado:")
print(" ", summary_json)
print(" ", summary_csv)
print("\nContenido JSON:")
print(json.dumps(summary, indent=2, ensure_ascii=False))